In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_E.xpt"
csv_file = DATA_DIR / "TRIGLY_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 TRIGLY_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


if df.shape != (3315, 6):

    raise ValueError(
        f"Unexpected TRIGLY_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: TRIGLY_E structure = "
    "3,315 rows x 6 columns."
)


# ============================================================
# 4. FIELD TYPES
# ============================================================

#
# ONLY SEQN is an identifier.
#
# Every other field is a sample weight or laboratory
# measurement and must remain source-decoded float.
#
# DO NOT:
#
# - convert measurements to Int64
# - round()
# - use float_format
# - trim legitimate ".0"
# - recalculate derived variables
#

INTEGER_COLUMNS = [
    "SEQN"
]


MEASUREMENT_COLUMNS = [
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI"
]


print(
    "\nIdentifier fields:",
    INTEGER_COLUMNS
)

print(
    "Lab/weight fields preserved AS IS:",
    MEASUREMENT_COLUMNS
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifacts:",
    f"{int(tiny_counts.sum()):,}"
)


print(
    "\nTiny values by variable:"
)

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 515
}

EXPECTED_TINY_TOTAL = 515


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny-artifact count."
    )


if int(
    true_zero_counts.sum()
) != 0:

    raise ValueError(
        "Unexpected true numeric zero already "
        "decoded in source XPT."
    )


print(
    "\nPASS: Exactly 515 tiny artifacts "
    "found, all in WTSAF2YR."
)


# ============================================================
# 7. RESTORE WTSAF2YR ZERO
# ============================================================

#
# CDC releases WTSAF2YR with range beginning at 0.
#
# Therefore:
#
# 5.397605346934028e-79 -> 0.0
#
# ONLY in WTSAF2YR.
#

mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


if int(
    mask.sum()
) != 515:

    raise ValueError(
        "Unexpected WTSAF2YR tiny count."
    )


df.loc[
    mask,
    "WTSAF2YR"
] = 0.0


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 515:

    raise ValueError(
        "WTSAF2YR restored-zero count differs."
    )


print(
    "PASS: WTSAF2YR 515 tiny values -> 0.0"
)


# ============================================================
# 8. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifact remains."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 9. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41479:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN 41479-51623, "
    "no missing, no duplicates."
)


# ============================================================
# 10. WTSAF2YR CDC CHECK
# ============================================================

s = df["WTSAF2YR"]


if int(
    s.notna().sum()
) != 3315:

    raise ValueError(
        "WTSAF2YR valid count differs."
    )


if int(
    s.isna().sum()
) != 0:

    raise ValueError(
        "Unexpected WTSAF2YR missing values."
    )


if s.min() != 0:

    raise ValueError(
        "WTSAF2YR minimum should be 0."
    )


if not np.isclose(
    s.max(),
    404655.623,
    rtol=0,
    atol=1e-12
):

    raise ValueError(
        f"Unexpected WTSAF2YR maximum: {s.max()}"
    )


if int(
    s.eq(0).sum()
) != 515:

    raise ValueError(
        "WTSAF2YR zero count differs."
    )


print(
    "\nPASS: WTSAF2YR"
)

print(
    "      Valid = 3,315"
)

print(
    "      Range = 0-404655.623"
)

print(
    "      Zeros = 515"
)


# ============================================================
# 11. LBXTR CDC CHECK
# ============================================================

#
# Triglyceride mg/dL
#
# CDC:
#
# Valid   = 3,099
# Missing = 216
# Range   = 12-2549
#
# KEEP AS FLOAT MEASUREMENT.
#

s = df["LBXTR"]


if not (
    int(s.notna().sum()) == 3099
    and
    int(s.isna().sum()) == 216
    and
    s.min() == 12
    and
    s.max() == 2549
):

    raise ValueError(
        "LBXTR differs from CDC."
    )


print(
    "\nPASS: LBXTR"
)

print(
    "      Valid   = 3,099"
)

print(
    "      Missing = 216"
)

print(
    "      Range   = 12-2549"
)


# ============================================================
# 12. LBDTRSI CDC CHECK
# ============================================================

#
# Triglyceride mmol/L
#
# CDC:
#
# Valid   = 3,099
# Missing = 216
# Range   = 0.135-28.778
#
# DO NOT recompute.
#

s = df["LBDTRSI"]


if not (
    int(s.notna().sum()) == 3099
    and
    int(s.isna().sum()) == 216
    and
    np.isclose(
        s.min(),
        0.135,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        28.778,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "LBDTRSI differs from CDC."
    )


print(
    "PASS: LBDTRSI"
)

print(
    "      Valid   = 3,099"
)

print(
    "      Missing = 216"
)

print(
    "      Range   = 0.135-28.778"
)


# ============================================================
# 13. LBDLDL CDC CHECK
# ============================================================

#
# LDL cholesterol mg/dL
#
# CDC:
#
# Valid   = 3,035
# Missing = 280
# Range   = 23-344
#
# This is a derived LAB measurement.
#
# DO NOT recalculate.
#

s = df["LBDLDL"]


if not (
    int(s.notna().sum()) == 3035
    and
    int(s.isna().sum()) == 280
    and
    s.min() == 23
    and
    s.max() == 344
):

    raise ValueError(
        "LBDLDL differs from CDC."
    )


print(
    "\nPASS: LBDLDL"
)

print(
    "      Valid   = 3,035"
)

print(
    "      Missing = 280"
)

print(
    "      Range   = 23-344"
)


# ============================================================
# 14. LBDLDLSI CDC CHECK
# ============================================================

#
# LDL cholesterol mmol/L
#
# CDC:
#
# Valid   = 3,035
# Missing = 280
# Range   = 0.595-8.896
#
# DO NOT recalculate.
#

s = df["LBDLDLSI"]


if not (
    int(s.notna().sum()) == 3035
    and
    int(s.isna().sum()) == 280
    and
    np.isclose(
        s.min(),
        0.595,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        8.896,
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "LBDLDLSI differs from CDC."
    )


print(
    "PASS: LBDLDLSI"
)

print(
    "      Valid   = 3,035"
)

print(
    "      Missing = 280"
)

print(
    "      Range   = 0.595-8.896"
)


# ============================================================
# 15. FRACTIONAL VALUE CHECK
# ============================================================

fractional_columns = {}


for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


EXPECTED_FRACTIONAL_COUNTS = {
    "WTSAF2YR": 2799,
    "LBDTRSI": 3099,
    "LBDLDLSI": 3008
}


print(
    "\n--- GENUINE FRACTIONAL VALUES ---"
)


for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


if fractional_columns != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n"
        f"Expected: {EXPECTED_FRACTIONAL_COUNTS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: Genuine fractional-value pattern "
    "matches attached XPT."
)


# ============================================================
# 16. WHOLE-NUMBER LAB MEASUREMENTS
# ============================================================

#
# LBXTR and LBDLDL happen to contain only
# mathematically whole results.
#
# They remain LAB MEASUREMENTS.
#
# So:
#
# LBXTR  120.0 -> 120.0
# LBDLDL 100.0 -> 100.0
#

for col in [
    "LBXTR",
    "LBDLDL"
]:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"Unexpected fractional value in {col}."
        )


print(
    "\nPASS: LBXTR and LBDLDL are whole-number "
    "measurements but will remain float-style."
)


# ============================================================
# 17. LEGITIMATE LAB .0 COUNTS
# ============================================================

#
# After WTSAF2YR tiny -> 0.0:
#
# WTSAF2YR = 516 values ending .0
# LBXTR = all 3,099 valid values
# LBDTRSI = none
# LBDLDL = all 3,035 valid values
# LBDLDLSI = 27
#

EXPECTED_MEASUREMENT_DOT_ZERO = {
    "WTSAF2YR": 516,
    "LBXTR": 3099,
    "LBDTRSI": 0,
    "LBDLDL": 3035,
    "LBDLDLSI": 27
}


print(
    "\n--- LEGITIMATE MEASUREMENT .0 ---"
)


for col, expected_count in (
    EXPECTED_MEASUREMENT_DOT_ZERO.items()
):

    actual_count = sum(
        str(float(value)).endswith(".0")
        for value in df[col].dropna()
    )


    print(
        f"{col:<10}: {actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected source .0 count.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate measurement .0 counts "
    "identified correctly."
)


# ============================================================
# 18. DERIVED VARIABLE NOTE
# ============================================================

#
# CDC documents:
#
# LBDTRSI   = LBXTR * 0.01129
#
# LBDLDLSI = LBDLDL * 0.02586
#
# LBDLDL itself is derived using the
# Friedewald calculation.
#
# DO NOT RECOMPUTE any of these.
#
# Released XPT values remain source of truth.
#

print(
    "\nINFO: LBDTRSI, LBDLDL and LBDLDLSI "
    "are released derived variables."
)

print(
    "      They will NOT be recalculated."
)


# ============================================================
# 19. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY:
#
# WTSAF2YR tiny -> 0.0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0.0


# ============================================================
# 20. CLEAN SEQN ONLY
# ============================================================

#
# SEQN:
#
# 41479.0 -> 41479
#
#
# Everything else remains FLOAT.
#

df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\nPASS: SEQN converted to clean integer formatting."
)

print(
    "PASS: All five laboratory/weight fields "
    "remain source-decoded floats."
)


# ============================================================
# 21. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

current_seqn = (
    df["SEQN"]
    .astype("Float64")
    .to_numpy(
        dtype=float,
        na_value=np.nan
    )
)


if not np.allclose(
    expected_df["SEQN"].to_numpy(),
    current_seqn,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SEQN numeric values changed."
    )


for col in MEASUREMENT_COLUMNS:

    if not np.allclose(
        expected_df[col].to_numpy(),
        df[col].to_numpy(),
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY WTSAF2YR zero restoration."
)


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
# round()
# float_format
# recalculation
# decimal quantization
#
# Let pandas preserve the actual decoded floats.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 23. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 24. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 6 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 26. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 27. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 3315:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 28. STRICT RAW CSV FIDELITY
# ============================================================

#
# SEQN:
#
# str(int(source_value))
#
#
# Lab/weight measurements:
#
# str(float(source_value))
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(example)


    raise ValueError(
        "TRIGLY_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected XPT-derived representation."
)


# ============================================================
# 29. UNWANTED SEQN .0 CHECK
# ============================================================

seqn_dot_zero_count = sum(
    row["SEQN"].endswith(".0")
    for row in raw_rows
    if row["SEQN"] != ""
)


print(
    "\nUnwanted SEQN '.0' values:",
    seqn_dot_zero_count
)


if seqn_dot_zero_count != 0:

    raise ValueError(
        "SEQN still contains .0."
    )


print(
    "PASS: No unwanted .0 in SEQN."
)


# ============================================================
# 30. LAB MEASUREMENT .0 CHECK
# ============================================================

for col, expected_count in (
    EXPECTED_MEASUREMENT_DOT_ZERO.items()
):

    actual_count = sum(
        row[col].endswith(".0")
        for row in raw_rows
        if row[col] != ""
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: legitimate .0 count changed.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


print(
    "PASS: Legitimate lab .0 formatting preserved."
)


# ============================================================
# 31. STRICT LAB RAW-MEASUREMENT CHECK
# ============================================================

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                float(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Triglyceride/LDL measurement "
        "representation changed."
    )


print(
    "PASS: Every lab/weight value preserved AS IS."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 34. FINAL ZERO CHECK
# ============================================================

final_zero_count = int(
    csv_df["WTSAF2YR"]
    .eq(0)
    .sum()
)


if final_zero_count != 515:

    raise ValueError(
        "Final WTSAF2YR zero count differs."
    )


print(
    "PASS: Final WTSAF2YR zero count = 515."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("TRIGLY_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41479-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "WTSAF2YR = 515"
)

print(
    "TOTAL     = 515"
)


print(
    "\nLab/weight measurements preserved AS IS:"
)

for col in MEASUREMENT_COLUMNS:
    print(col)


print(
    "\nGenuine fractional fields:"
)

for col, count in fractional_columns.items():

    print(
        f"{col:<10}: {count:,}"
    )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Lab measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted SEQN .0:",
    seqn_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "TRIGLY_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 TRIGLY_E XPT -> CSV
Rows: 3,315
Columns: 6
PASS: TRIGLY_E structure = 3,315 rows x 6 columns.

Identifier fields: ['SEQN']
Lab/weight fields preserved AS IS: ['WTSAF2YR', 'LBXTR', 'LBDTRSI', 'LBDLDL', 'LBDLDLSI']

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 515

Tiny values by variable:
WTSAF2YR    515

PASS: Exactly 515 tiny artifacts found, all in WTSAF2YR.
PASS: WTSAF2YR 515 tiny values -> 0.0
Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

PASS: SEQN 41479-51623, no missing, no duplicates.

PASS: WTSAF2YR
      Valid = 3,315
      Range = 0-404655.623
      Zeros = 515

PASS: LBXTR
      Valid   = 3,099
      Missing = 216
      Range   = 12-2549
PASS: LBDTRSI
      Valid   = 3,099
      Missing = 216
      Range   = 0.135-28.778

PASS: LBDLDL
      Valid   = 3,035
      Missing = 280
      Range   = 23-344
PASS: LBDLDLSI
      Valid   = 3,035
      Missing = 280
      Range   = 0.595-8.896

--- GENUINE FRACTIONAL VALUES ---
W